# basic CytoVI workflow

In [ ]:
import cytovi
import readfcs
import scanpy as sc
import matplotlib.pyplot as plt

In [ ]:
# read data
adata_batch1 = readfcs.read('../data/raw/Spectral flow/Nunez/For Chiquito/Raw_100000/batch1')
adata_batch1.layers['raw'] = adata_batch1.X.copy()

adata_batch2 = readfcs.read('../data/raw/Spectral flow/Nunez/For Chiquito/Raw_100000/batch2')
adata_batch2.layers['raw'] = adata_batch2.X.copy()

In [ ]:
# preprocess
cytovi.pp.arcsinh(adata_batch1, global_scaling_factor=2000)
cytovi.pp.scale(adata_batch1)

cytovi.pp.arcsinh(adata_batch2, global_scaling_factor=2000)
cytovi.pp.scale(adata_batch2)

In [ ]:
# merge batches
adata = cytovi.pp.merge_batches([adata_batch1, adata_batch2])

In [ ]:
# do some plotting
cytovi.pl.histogram(adata, marker = 'all', groupby='batch', layer_key='transformed')
cytovi.pl.biaxial(adata, marker_x = 'CD3', marker_y = 'CD4', groupby='batch', layer_key='transformed')

In [ ]:
# train model
cytovi.CytoVI.setup_anndata(adata, layer='scaled', batch_key='batch')
model = cytovi.CytoVI(adata)
model.train()

In [ ]:
# save model on disk for later use
model_path = '../saved_models/'
model.save(f'{model_path}my_cytovi_model', overwrite=True)

In [ ]:
# plot training dynamics
plt.subplot(1, 2, 1)
plt.plot(model.history['elbo_train'])
plt.xlabel('epochs')
plt.ylabel('elbo_train')

plt.subplot(1, 2, 2)
plt.plot(model.history['elbo_validation'])
plt.xlabel('epochs')
plt.ylabel('elbo_validation')

In [ ]:
# compute umap and cluster CytoVI latent space
adata.obsm["X_CytoVI"] = model.get_latent_representation()
sc.pp.neighbors(adata, use_rep="X_CytoVI")
sc.tl.umap(adata)
sc.tl.leiden(adata, key_added="leiden_CytoVI")

In [ ]:
# plot data
sc.pl.umap(adata, color=["leiden_CytoVI", "batch"])
sc.pl.matrixplot(adata, var_names=adata.var_names, groupby="leiden_CytoVI")

In [ ]:
# annotate cells
annot_dict = {
    '0': 'Granulocytes',
    '1': 'Monocytes',
    '2': 'Monocytes',
    '3': 'B cells',
    '4': 'T cells',
    '5': 'NK cells'
}
adata.obs['cell_type'] = adata.obs['leiden_CytoVI'].map(annot_dict)
sc.pl.umap(adata, color = ['batch', 'leiden_CytoVI', 'cell_type'])

In [ ]:
# get imputed expression, note: by default we return the mean across all batches, you can change this by setting the transform_batch argument
adata.layers['imputed'] = model.get_normalized_expression(adata, n_samples = 10)

# visualize imputed expression
sc.pl.umap(adata, color = ['CD3', 'CD4', 'CD8', 'CD19', 'CD56'], layer='imputed')